In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.s4lake_gold"
CHAVE = "cod_fatura"
COL_EMISSAO = "data_base"

In [0]:
def comentar_tabela(tabela, comentario_tabela, comentarios_colunas):
    nome = f"{GOLD}.{tabela}"

    def texto_sql(texto):
        # Escapa aspas simples para não fecharem a string do SQL antes da hora
        return texto.replace("'", "\\'")

    spark.sql(f"COMMENT ON TABLE {nome} IS '{texto_sql(comentario_tabela)}'")

    colunas_tabela = spark.table(nome).columns
    for coluna, texto in comentarios_colunas.items():
        if coluna in colunas_tabela:
            spark.sql(f"ALTER TABLE {nome} ALTER COLUMN {coluna} COMMENT '{texto_sql(texto)}'")

    sem_comentario = [c for c in colunas_tabela if c not in comentarios_colunas]
    nao_encontradas = [c for c in comentarios_colunas if c not in colunas_tabela]
    print(f"{tabela}: {len(colunas_tabela) - len(sem_comentario)} de {len(colunas_tabela)} colunas comentadas")
    print("  Colunas sem comentário:", sem_comentario)
    print("  Comentários sem coluna:", nao_encontradas)

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

In [0]:
fato_titulos = spark.table(f"{GOLD}.fato_titulos")
fato_titulos.printSchema()

print("linhas:", fato_titulos.count())
print("chaves únicas:", fato_titulos.select(CHAVE).distinct().count())

In [0]:
atual = fato_titulos.select(
    F.col(CHAVE),
    F.col("cod_cliente"),
    F.col(COL_EMISSAO).alias("emissao"),
)

hist = fato_titulos.select(
    F.col(CHAVE).alias("chave_h"),
    F.col("cod_cliente").alias("cod_cliente_h"),
    F.col("data_vencimento").alias("data_vencimento_h"),
    F.col("data_pagamento").alias("data_pagamento_h"),
    F.col("dias_atraso").alias("dias_atraso_h"),
)

In [0]:
# Situações do histórico conhecidas na emissão do título atual (mutuamente exclusivas)
pago_antes = F.col("data_pagamento_h") < F.col("emissao")

aberto_vencido = (
    (F.col("data_vencimento_h") < F.col("emissao"))
    & (F.col("data_pagamento_h").isNull() | (F.col("data_pagamento_h") >= F.col("emissao")))
)

pares = atual.join(
    hist,
    (F.col("cod_cliente") == F.col("cod_cliente_h"))
    & (F.col(CHAVE) != F.col("chave_h"))          # nunca o próprio título
    & (pago_antes | aberto_vencido),              # condição de data dentro do join
    how="left",
)

In [0]:
pares = pares.withColumn(
    "atraso_conhecido",
    F.when(pago_antes, F.greatest(F.col("dias_atraso_h"), F.lit(0)))          # atraso completo
     .when(aberto_vencido, F.datediff(F.col("emissao"), F.col("data_vencimento_h")))  # atraso até a emissão
)

In [0]:
features_hist = (
    pares
    .groupBy(CHAVE)
    .agg(
        F.round(F.avg("atraso_conhecido"), 2).alias("atraso_medio_hist"),
        F.count("atraso_conhecido").alias("qtd_titulos_hist"),   # auxiliar para validar
    )
)

In [0]:
print("linhas:", features_hist.count())
print("chaves únicas:", features_hist.select(CHAVE).distinct().count())
print("sem histórico:", features_hist.filter(F.col("atraso_medio_hist").isNull()).count())
print("incoerentes:", features_hist.filter(
    F.col("atraso_medio_hist").isNull() & (F.col("qtd_titulos_hist") > 0)
).count())

In [0]:
(
    features_hist
    .join(atual, on=CHAVE)
    .filter(F.col("cod_cliente") == "0000100178")
    .orderBy(F.col("emissao").desc())
    .select(CHAVE, "emissao", "qtd_titulos_hist", "atraso_medio_hist")
    .show(10)
)

In [0]:
salvar(features_hist, GOLD, "features_historico_cliente")

In [0]:
comentar_tabela(
    "features_historico_cliente",
    "Features de histórico do cliente para o modelo de risco de atraso. Uma linha por título (cod_fatura). "
    "Calculadas point-in-time: cada título enxerga apenas os outros títulos do mesmo cliente que já estavam "
    "pagos ou vencidos na sua data de emissão (data_base), sem informação posterior.",
    {
        "cod_fatura": "Número da fatura que originou o título; chave da tabela e da junção com a fato_titulos.",
        "atraso_medio_hist": "Atraso médio, em dias, dos títulos do mesmo cliente conhecidos na emissão. "
                             "Pagos antes da emissão entram com o atraso final (negativos zerados); vencidos "
                             "e em aberto na emissão entram com o atraso contado até a emissão. Nulo quando "
                             "o cliente não tem nenhum título pago ou vencido na emissão.",
        "qtd_titulos_hist": "Quantidade de títulos do mesmo cliente usados no cálculo do atraso_medio_hist. "
                            "Zero quando não há histórico; indica a confiança da média.",
    },
)

In [0]:
%sql
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_fatura) AS chaves_unicas,
  SUM(CASE WHEN atraso_medio_hist IS NULL THEN 1 ELSE 0 END) AS sem_historico,
  MAX(CASE WHEN cod_fatura = '0090005631' THEN qtd_titulos_hist END)  AS qtd_178,
  MAX(CASE WHEN cod_fatura = '0090005631' THEN atraso_medio_hist END) AS atraso_178
FROM workspace.s4lake_gold.features_historico_cliente